In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login

# 2. Authenticate with Hugging Face
# You will be prompted to paste your token here
print("Logging into Hugging Face...")
login()


Logging into Hugging Face...


In [ ]:
# ==============================================================================
# Internal State Probing
# Model: Meta-Llama-3-8B-Instruct
# ==============================================================================

# 1. Install Hugging Face and PyTorch libraries (Run this first if you haven't)
# !pip install transformers accelerate bitsandbytes torch datasets

# import torch
# from transformers import AutoModelForCausalLM, AutoTokenizer
# from huggingface_hub import login

# # 2. Authenticate with Hugging Face
# # You will be prompted to paste your token here
# print("Logging into Hugging Face...")
# login()

# 3. Model Configuration
model_id = "meta-llama/Meta-Llama-3.1-8B-Instruct"

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Llama-3 doesn't have a default pad token, so we set it to the eos token to avoid warnings
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model {model_id} into GPU memory...")
# We load in bfloat16 to save memory and optimize for your GPU,
# and use device_map="auto" to automatically manage VRAM distribution.
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    # DO NOT Change the OUTPUT HIDDEN STATES
    # It tells PyTorch to expose the internal mathematical states
    output_hidden_states=True
)

print("\nModel successfully loaded!")
print(f"Total parameters: {model.num_parameters():,}")
print(f"Number of hidden layers (Transformer blocks): {model.config.num_hidden_layers}")

Loading tokenizer for meta-llama/Meta-Llama-3.1-8B-Instruct...


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


Model successfully loaded!
Total parameters: 8,030,261,248
Number of hidden layers (Transformer blocks): 32


In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ==============================================================================
# 1. Load the Small Language Model (Qwen2.5-3B-Instruct)
# ==============================================================================
model_id = "Qwen/Qwen2.5-3B-Instruct"

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print(f"Loading {model_id} into GPU memory...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    output_hidden_states=True # Expose the SLM's internal states
)

print(f"\nModel successfully loaded!")
print(f"Total parameters: {model.num_parameters():,}")
print(f"Hidden dimension size: {model.config.hidden_size}") # Should be 2048
print(f"Number of layers: {model.config.num_hidden_layers}") # Should be 36

# ==============================================================================
# 2. Data Standardization Function (HaluEval)
# ==============================================================================
def load_and_standardize_task(task_name):
    print(f"\n{'='*50}")
    print(f" LOADING {task_name.upper()} DATASET ")
    print(f"{'='*50}")
    dataset = load_dataset("pminervini/HaluEval", task_name)
    df = pd.DataFrame(dataset['data'])

    if task_name == "qa":
        col_context, col_right, col_hallu = 'question', 'right_answer', 'hallucinated_answer'
    elif task_name == "dialogue":
        col_context, col_right, col_hallu = 'dialogue_history', 'right_response', 'hallucinated_response'
    elif task_name == "summarization":
        col_context, col_right, col_hallu = 'document', 'right_summary', 'hallucinated_summary'

    df_factual = df[[col_context, col_right]].copy()
    df_factual.columns = ['context', 'response']
    df_factual['label'] = 0

    df_hallucinated = df[[col_context, col_hallu]].copy()
    df_hallucinated.columns = ['context', 'response']
    df_hallucinated['label'] = 1

    df_combined = pd.concat([df_factual, df_hallucinated]).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_combined

# ==============================================================================
# 3. Extraction & Evaluation Loop
# ==============================================================================
tasks = ["qa", "dialogue", "summarization"]

for task in tasks:
    # Loading the full 10,000-row dataset for the A/B test
    df_task = load_and_standardize_task(task)

    X_hidden_states = []
    y_labels = df_task['label'].tolist()

    print(f"Extracting SLM states for {len(df_task)} rows of {task.upper()}...")
    model.eval()

    for index, row in tqdm(df_task.iterrows(), total=len(df_task)):
        # Qwen handles standard text formatting beautifully
        text = f"Context: {row['context']}\nOutput: {row['response']}"

        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        # Extract the final token state (2048 dimensions)
        final_token_state = outputs.hidden_states[-1][0, -1, :].cpu().float().numpy()
        X_hidden_states.append(final_token_state)

    X_features = np.array(X_hidden_states)
    y_targets = np.array(y_labels)

    print(f"\nEvaluating {task.upper()}...")
    X_train, X_test, y_train, y_test = train_test_split(X_features, y_targets, test_size=0.2, random_state=42)

    # Adding max_iter=2000 to help prevent the convergence warning we saw earlier
    clf = LogisticRegression(max_iter=2000, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    print(f"\n--- RESULTS FOR {task.upper()} (Qwen 3B Model) ---")
    print("\n=== CLASSIFICATION REPORT ===")
    print(classification_report(y_test, y_pred, target_names=['Factual (0)', 'Hallucination (1)']))

    print("=== CONTINUOUS UNCERTAINTY METRICS ===")
    print(f"AUROC: {roc_auc_score(y_test, y_prob):.4f}")
    print(f"AUPRC: {average_precision_score(y_test, y_prob):.4f}\n")

Loading tokenizer for Qwen/Qwen2.5-3B-Instruct...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

summarization/data-00000-of-00001.parque(…):   0%|          | 0.00/28.0M [00:00<?, ?B/s]

Generating data split:   0%|          | 0/10000 [00:00<?, ? examples/s]

Extracting SLM states for 20000 rows of SUMMARIZATION...


  0%|          | 38/20000 [01:33<13:34:48,  2.45s/it]


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm import tqdm

# ==============================================================================
# 1. Load the Small Language Model (Llama-3.2-3B-Instruct)
# ==============================================================================
# Relying on the Hugging Face login already executed in the notebook environment
model_id = "meta-llama/Llama-3.2-3B-Instruct"

print(f"Loading tokenizer for {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading {model_id} into GPU memory...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    output_hidden_states=True # Expose the SLM's internal states
)

print(f"Total parameters: {model.num_parameters():,}")
print(f"Hidden dimension size: {model.config.hidden_size}") # Should be 3072

# ==============================================================================
# 2. Data Standardization Function (HaluEval)
# ==============================================================================
def load_and_standardize_task(task_name):
    print(f"\n{'='*50}")
    print(f" LOADING {task_name.upper()} DATASET ")
    print(f"{'='*50}")
    dataset = load_dataset("pminervini/HaluEval", task_name)
    df = pd.DataFrame(dataset['train'])

    if task_name == "qa":
        col_context, col_right, col_hallu = 'question', 'right_answer', 'hallucinated_answer'
    elif task_name == "dialogue":
        col_context, col_right, col_hallu = 'dialogue_history', 'right_response', 'hallucinated_response'
    elif task_name == "summarization":
        col_context, col_right, col_hallu = 'document', 'right_summary', 'hallucinated_summary'

    df_factual = df[[col_context, col_right]].copy()
    df_factual.columns = ['context', 'response']
    df_factual['label'] = 0

    df_hallucinated = df[[col_context, col_hallu]].copy()
    df_hallucinated.columns = ['context', 'response']
    df_hallucinated['label'] = 1

    df_combined = pd.concat([df_factual, df_hallucinated]).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_combined

# ==============================================================================
# 3. Extraction & Evaluation Loop
# ==============================================================================
tasks = ["qa", "dialogue", "summarization"]

for task in tasks:
    df_task = load_and_standardize_task(task)

    X_hidden_states = []
    y_labels = df_task['label'].tolist()

    print(f"Extracting SLM states for {len(df_task)} rows of {task.upper()}...")
    model.eval()

    for index, row in tqdm(df_task.iterrows(), total=len(df_task)):
        text = f"Context: {row['context']}\nOutput: {row['response']}"

        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        # Extract the final token state (now 3072 dimensions instead of 4096)
        final_token_state = outputs.hidden_states[-1][0, -1, :].cpu().float().numpy()
        X_hidden_states.append(final_token_state)

    X_features = np.array(X_hidden_states)
    y_targets = np.array(y_labels)

    print(f"\nEvaluating {task.upper()}...")
    X_train, X_test, y_train, y_test = train_test_split(X_features, y_targets, test_size=0.2, random_state=42)

    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_prob = clf.predict_proba(X_test)[:, 1]

    print(f"\n--- RESULTS FOR {task.upper()} (3B Model) ---")
    print("\n=== CLASSIFICATION REPORT ===")
    print(classification_report(y_test, y_pred, target_names=['Factual (0)', 'Hallucination (1)']))

    print("=== CONTINUOUS UNCERTAINTY METRICS ===")
    print(f"AUROC: {roc_auc_score(y_test, y_prob):.4f}")
    print(f"AUPRC: {average_precision_score(y_test, y_prob):.4f}\n")

Loading tokenizer for meta-llama/Llama-3.2-3B-Instruct...


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct.
403 Client Error. (Request ID: Root=1-69cca112-2c9e28107a2ccbec3cc2f0cb;70edefee-85a0-423d-a3a3-5e0d2917b1a7)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.2-3B-Instruct is restricted and you are not in the authorized list. Visit https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct to ask for access.

In [ ]:
import pandas as pd
import numpy as np
import torch
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from tqdm import tqdm

# ==============================================================================
# 1. Data Standardization Function
# ==============================================================================
def load_and_standardize_task(task_name):
    print(f"\n{'='*50}")
    print(f" LOADING {task_name.upper()} DATASET ")
    print(f"{'='*50}")
    dataset = load_dataset("pminervini/HaluEval", task_name)
    df = pd.DataFrame(dataset['data'])

    # Standardize column names based on the task
    if task_name == "qa":
        col_context, col_right, col_hallu = 'question', 'right_answer', 'hallucinated_answer'
    elif task_name == "dialogue":
        col_context, col_right, col_hallu = 'dialogue_history', 'right_response', 'hallucinated_response'
    elif task_name == "summarization":
        col_context, col_right, col_hallu = 'document', 'right_summary', 'hallucinated_summary'

    # Create Factual (0) and Hallucinated (1) splits
    df_factual = df[[col_context, col_right]].copy()
    df_factual.columns = ['context', 'response']
    df_factual['label'] = 0

    df_hallucinated = df[[col_context, col_hallu]].copy()
    df_hallucinated.columns = ['context', 'response']
    df_hallucinated['label'] = 1

    # Combine and shuffle
    df_combined = pd.concat([df_factual, df_hallucinated]).sample(frac=1, random_state=42).reset_index(drop=True)
    return df_combined

# ==============================================================================
# 2. Full Dataset Extraction & Evaluation Loop
# ==============================================================================
tasks = ["qa", "dialogue", "summarization"]

for task in tasks:
    # We are loading the ENTIRE dataset now
    df_task = load_and_standardize_task(task)

    X_hidden_states = []
    y_labels = df_task['label'].tolist()

    print(f"Extracting Llama-3 states for {len(df_task)} rows of {task.upper()}...")
    model.eval()

    # tqdm will now show progress out of the full dataset size
    for index, row in tqdm(df_task.iterrows(), total=len(df_task)):
        text = f"Context: {row['context']}\nOutput: {row['response']}"

        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=2048).to(model.device)

        with torch.no_grad():
            outputs = model(**inputs)

        final_token_state = outputs.hidden_states[-1][0, -1, :].cpu().float().numpy()
        X_hidden_states.append(final_token_state)

    X_features = np.array(X_hidden_states)
    y_targets = np.array(y_labels)

    # --- Train Logistic Regression & Get Metrics ---
    print(f"\nEvaluating {task.upper()}...")
    X_train, X_test, y_train, y_test = train_test_split(X_features, y_targets, test_size=0.2, random_state=42)


    # ... [your extraction code above] ...

    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X_train, y_train)

    # 1. Get the hard binary predictions (0 or 1) for Precision/Recall
    y_pred = clf.predict(X_test)

    # 2. Get the continuous probabilities for AUROC/AUPRC
    y_prob = clf.predict_proba(X_test)[:, 1]

    print(f"\n--- RESULTS FOR {task.upper()} ---")

    # 3. Print the Precision, Recall, and F1-Score
    print("\n=== CLASSIFICATION REPORT ===")
    print(classification_report(y_test, y_pred, target_names=['Factual (0)', 'Hallucination (1)']))
    # 4. Print the Continuous Metrics
    print("=== CONTINUOUS UNCERTAINTY METRICS ===")
    print(f"AUROC: {roc_auc_score(y_test, y_prob):.4f}")
    print(f"AUPRC: {average_precision_score(y_test, y_prob):.4f}\n")


 LOADING QA DATASET 
Extracting Llama-3 states for 20000 rows of QA...


100%|██████████| 20000/20000 [13:27<00:00, 24.77it/s]



Evaluating QA...

--- RESULTS FOR QA ---

=== CLASSIFICATION REPORT ===
                   precision    recall  f1-score   support

      Factual (0)       0.97      0.97      0.97      2055
Hallucination (1)       0.97      0.97      0.97      1945

         accuracy                           0.97      4000
        macro avg       0.97      0.97      0.97      4000
     weighted avg       0.97      0.97      0.97      4000

=== CONTINUOUS UNCERTAINTY METRICS ===
AUROC: 0.9875
AUPRC: 0.9910


 LOADING DIALOGUE DATASET 
Extracting Llama-3 states for 20000 rows of DIALOGUE...


100%|██████████| 20000/20000 [13:46<00:00, 24.19it/s]



Evaluating DIALOGUE...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



--- RESULTS FOR DIALOGUE ---

=== CLASSIFICATION REPORT ===
                   precision    recall  f1-score   support

      Factual (0)       0.82      0.79      0.80      2055
Hallucination (1)       0.79      0.81      0.80      1945

         accuracy                           0.80      4000
        macro avg       0.80      0.80      0.80      4000
     weighted avg       0.80      0.80      0.80      4000

=== CONTINUOUS UNCERTAINTY METRICS ===
AUROC: 0.8897
AUPRC: 0.8864


 LOADING SUMMARIZATION DATASET 
Extracting Llama-3 states for 20000 rows of SUMMARIZATION...


100%|██████████| 20000/20000 [29:44<00:00, 11.20it/s]



Evaluating SUMMARIZATION...

--- RESULTS FOR SUMMARIZATION ---

=== CLASSIFICATION REPORT ===
                   precision    recall  f1-score   support

      Factual (0)       0.98      0.99      0.99      2055
Hallucination (1)       0.99      0.98      0.98      1945

         accuracy                           0.98      4000
        macro avg       0.98      0.98      0.98      4000
     weighted avg       0.98      0.98      0.98      4000

=== CONTINUOUS UNCERTAINTY METRICS ===
AUROC: 0.9995
AUPRC: 0.9995

